# Model Development for BNPL Risk Prediction

This notebook builds on the earlier data cleaning and feature engineering stages of the project. Using the processed and engineered datasets, we develop predictive models for BNPL repayment risk.

We begin with a logistic regression model as a baseline due to its interpretability, and then extend the analysis to a Bayesian framework to quantify uncertainty in model parameters and predictions.

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load processed dataset from GitHub
url = "https://raw.githubusercontent.com/samarapires-ml/Buy-Now-Pay-Later--BNPL--Risk-Prediction/main/data/processed/clean_full_data.csv"
df = pd.read_csv(url)

# Preview data
df.head()

,Customer_Age,Annual_Income,Credit_Score,Purchase_Amount,Checkout_Time_Seconds,Gender_Male,Gender_Non-Binary,Purchase_Category_Electronics,Purchase_Category_Fashion,Purchase_Category_Groceries,...,BNPL_Provider_Klarna,BNPL_Provider_Sezzle,Device_Type_Mobile,Device_Type_Tablet,Connection_Type_VPN,Connection_Type_WiFi,Browser_Edge,Browser_Firefox,Browser_Safari,target
0,1.110297,-1.303034,-1.388519,-0.445029,-0.199028,True,False,False,False,False,...,False,True,False,True,False,True,False,True,False,0
1,0.371122,0.096571,-1.382224,-0.531422,-0.632426,True,False,False,False,True,...,False,False,True,False,False,True,False,True,False,0
2,-0.663723,0.422711,0.355240,1.482538,-0.120229,True,False,False,False,False,...,False,True,False,False,False,True,False,False,False,0
3,1.405967,0.778516,-0.651985,-0.627730,1.514864,True,False,False,True,False,...,False,True,True,False,False,False,False,False,False,0
4,-1.181145,-1.311090,-0.450540,1.821031,-1.065824,True,False,False,False,False,...,True,False,True,False,False,False,False,False,False,1


In [3]:
# Check all columns
df.columns

Index(['Customer_Age', 'Annual_Income', 'Credit_Score', 'Purchase_Amount',
       'Checkout_Time_Seconds', 'Gender_Male', 'Gender_Non-Binary',
       'Purchase_Category_Electronics', 'Purchase_Category_Fashion',
       'Purchase_Category_Groceries', 'Purchase_Category_Home & Furniture',
       'Purchase_Category_Travel', 'BNPL_Provider_Afterpay',
       'BNPL_Provider_Klarna', 'BNPL_Provider_Sezzle', 'Device_Type_Mobile',
       'Device_Type_Tablet', 'Connection_Type_VPN', 'Connection_Type_WiFi',
       'Browser_Edge', 'Browser_Firefox', 'Browser_Safari', 'target'],
      dtype='object')

In [4]:
# Separate features and target
X = df.drop(columns=["target"])
y = df["target"]

# Check shapes
print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (50000, 22)
y shape: (50000,)


In [5]:
# Load train-test data from feature engineering output

X_train = pd.read_csv("https://raw.githubusercontent.com/samarapires-ml/Buy-Now-Pay-Later--BNPL--Risk-Prediction/main/data/processed/X_train.csv")
X_test = pd.read_csv("https://raw.githubusercontent.com/samarapires-ml/Buy-Now-Pay-Later--BNPL--Risk-Prediction/main/data/processed/X_test.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/samarapires-ml/Buy-Now-Pay-Later--BNPL--Risk-Prediction/main/data/processed/y_train.csv").squeeze()
y_test = pd.read_csv("https://raw.githubusercontent.com/samarapires-ml/Buy-Now-Pay-Later--BNPL--Risk-Prediction/main/data/processed/y_test.csv").squeeze()

# Check shapes
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (40000, 33)
X_test: (10000, 33)
y_train: (40000,)
y_test: (10000,)


## Baseline Model: Logistic Regression

We begin with a logistic regression model as a baseline due to its interpretability. Logistic regression models the probability of default as a function of input features using the logistic function:

P(Y = 1 | X) = 1 / (1 + exp(-(β₀ + βᵀX)))

This provides a clear understanding of how each feature influences the likelihood of a delayed or missed payment.

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

# Initialize model
log_model = LogisticRegression(max_iter=1000)

# Train model
log_model.fit(X_train, y_train)

# Predictions
y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:, 1]

# Evaluation
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

Accuracy: 0.7792
ROC-AUC: 0.7043233356840761

Classification Report:
               precision    recall  f1-score   support

           0       0.78      0.99      0.87      7642
           1       0.70      0.11      0.19      2358

    accuracy                           0.78     10000
   macro avg       0.74      0.55      0.53     10000
weighted avg       0.76      0.78      0.71     10000



## Model Evaluation and Interpretation

The logistic regression model achieves an accuracy of approximately 78% and a ROC-AUC score of around 0.70, indicating moderate predictive performance.

However, a closer look at the classification report reveals a significant imbalance in predictive performance across classes. The model achieves a very high recall for non-default cases (class 0), but a very low recall for default cases (class 1), indicating that it fails to correctly identify many high-risk customers.

This suggests that the model is biased toward predicting the majority class and may not be suitable for real-world risk prediction without further adjustment. In the context of BNPL systems, failing to identify defaulting users can have significant financial implications.

This limitation motivates the need for more robust modeling approaches, such as Bayesian methods, which provide uncertainty estimates and can improve interpretability.

# Feature Importance - Logistic Regression Coefficients 

In [7]:
# Extract coefficients
coef_df = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": log_model.coef_[0]
})

# Sort by importance
coef_df = coef_df.sort_values(by="Coefficient", ascending=False)

# Show top positive (increase risk)
print("Top features increasing default risk:\n")
print(coef_df.head(10))

# Show top negative (decrease risk)
print("\nTop features decreasing default risk:\n")
print(coef_df.tail(10))

Top features increasing default risk:

                          Feature  Coefficient
25            Connection_Type_VPN     1.117361
20         BNPL_Provider_Afterpay     0.145606
23             Device_Type_Mobile     0.050142
16      Purchase_Category_Fashion     0.043879
15  Purchase_Category_Electronics     0.015247
11              Purchase_Amount.1     0.006088
2                 Purchase_Amount     0.006088
8                  Customer_Age.1     0.004157
4                    Customer_Age     0.004157
27                   Browser_Edge     0.003045

Top features decreasing default risk:

                               Feature  Coefficient
29                      Browser_Safari    -0.023436
17         Purchase_Category_Groceries    -0.030875
24                  Device_Type_Tablet    -0.040560
14                   Gender_Non-Binary    -0.042653
0                        Annual_Income    -0.046693
9                      Annual_Income.1    -0.046693
26                Connection_Type_WiFi  

## Feature Importance Analysis

The logistic regression coefficients provide insight into how different features influence the probability of default.

Among the features that increase default risk, the most significant predictors include the use of a VPN connection, transactions made through the Afterpay BNPL provider, and purchases made on mobile devices. The strong positive coefficient for VPN usage may indicate potentially higher-risk or less traceable transaction environments. Similarly, certain BNPL providers may be associated with different customer risk profiles. Mobile device usage may reflect faster or more impulsive purchasing behavior.

On the other hand, features such as higher credit scores and higher annual income are strongly associated with a lower probability of default, as expected. These variables are traditional indicators of financial stability. Additionally, certain purchase categories and connection types (such as WiFi) are associated with reduced risk.

Overall, the model provides interpretable insights into how both behavioral and financial factors contribute to BNPL repayment risk.

# Bayesian Model 

In [8]:
!pip install pymc arviz

In [9]:
import pymc as pm
import arviz as az

print("PyMC version:", pm.__version__)
print("ArviZ version:", az.__version__)

WARNING (pytensor.configdefaults): g++ not available, if using conda: `conda install gxx`
WARNING (pytensor.configdefaults): g++ not detected!  PyTensor will be unable to compile C-implementations and will default to Python. Performance may be severely degraded. To remove this warning, set PyTensor flags cxx to an empty string.


PyMC version: 5.25.1
ArviZ version: 0.23.4


## Bayesian Logistic Regression (PyMC)

In addition to the classical logistic regression model, we implement a Bayesian logistic regression model using PyMC. 

This approach allows us to:
- Quantify uncertainty in model parameters
- Obtain probability distributions instead of point estimates
- Improve interpretability of feature effects

We model the probability of default using a logistic function with normally distributed priors on coefficients.

In [10]:
import pymc as pm
import numpy as np
import pandas as pd
import arviz as az

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

In [11]:
# Making sure X_train, X_test, y_train, y_test already exist

# Select top features (IMPORTANT for speed)
selected_features = X_train.columns[:10]

X_train_small = X_train[selected_features]
X_test_small = X_test[selected_features]

# Standardize features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_small)
X_test_scaled = scaler.transform(X_test_small)

# Convert target to numpy
y_train_array = y_train.values
y_test_array = y_test.values

In [1]:
print("Kernel restarted successfully")

Kernel restarted successfully


In [ ]:
with pm.Model() as model:
    
    intercept = pm.Normal("intercept", mu=0, sigma=5)
    
    coefficients = pm.Normal(
        "coefficients", 
        mu=0, 
        sigma=2, 
        shape=X_train_scaled.shape[1]
    )
    
    logits = intercept + pm.math.dot(X_train_scaled, coefficients)
    
    likelihood = pm.Bernoulli(
        "likelihood",
        logit_p=logits,
        observed=y_train_array
    )
    
    trace = pm.sample(
        draws=200,        # 🔥 small = fast
        tune=200,
        target_accept=0.9,
        cores=1,          # 🔥 NO multiprocessing (safe)
        random_seed=42
    )

Initializing NUTS using jitter+adapt_diag...
c:\Users\91869\anaconda3\envs\bnpl_env\lib\site-packages\pytensor\scalar\basic.py:2094: RuntimeWarning: invalid value encountered in divide
  return x / y
Sequential sampling (2 chains in 1 job)
NUTS: [intercept, coefficients]


Output()